In [3]:
from snowflake.snowpark import Session
import pandas as pd
from dotenv import load_dotenv
import os

load_dotenv()

connection_params = {
    "account": os.getenv("SNOWFLAKE_ACCOUNT"),
    "user": os.getenv("SNOWFLAKE_USER"),
    "password": os.getenv("SNOWFLAKE_PASSWORD"),
    "role": "ACCOUNTADMIN",
    "warehouse": "GOLD_WH",
    "database": "GOLD_PROJECT",
    "schema": "PROCESSED"
}

session = Session.builder.configs(connection_params).create()

df = session.table("MASTER_WITH_INDICATORS").to_pandas()

df.head()


,DATE,GOLD_CLOSE,GOLD_OPEN,GOLD_HIGH,GOLD_LOW,GOLD_VOLUME,DXY,CRUDE_OIL,SP500,USDINR,...,MACD_SIGNAL,MACD_HIST,BB_LOWER,BB_MIDDLE,BB_UPPER,ATR,STOCH_K,STOCH_D,MOMENTUM10,ROC
0,2000-08-30,273.899994,273.899994,273.899994,273.899994,0,98.935997,59.23,6705.120117,90.124001,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
1,2000-08-31,278.299988,274.799988,278.299988,274.799988,0,98.935997,59.23,6705.120117,90.124001,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
2,2000-09-01,277.000000,277.000000,277.000000,277.000000,0,98.935997,59.23,6705.120117,90.124001,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
3,2000-09-05,275.799988,275.799988,275.799988,275.799988,2,98.935997,59.23,6705.120117,90.124001,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
4,2000-09-06,274.200012,274.200012,274.200012,274.200012,0,98.935997,59.23,6705.120117,90.124001,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0


In [4]:
df['DATE'] = pd.to_datetime(df['DATE'])
df = df.sort_values('DATE').reset_index(drop=True)


In [8]:
features = [
    'GOLD_CLOSE',      # Target
    'DXY',
    'CRUDE_OIL',
    'SP500',
    'USDINR',
    'CPI',
    'US10Y',
    'VIX',

    # Technical Indicators
    'RSI14',
    'EMA12',
    'EMA26',
    'SMA20',
    'SMA50',
    'BB_MIDDLE',
    'ATR',
    'MOMENTUM10'
]
df_model = df[features].copy()
df_model.head()


,GOLD_CLOSE,DXY,CRUDE_OIL,SP500,USDINR,CPI,US10Y,VIX,RSI14,EMA12,EMA26,SMA20,SMA50,BB_MIDDLE,ATR,MOMENTUM10
0,273.899994,98.935997,59.23,6705.120117,90.124001,323.364,4.04,16.43,0.000000,0.0,0.0,0.0,0.0,0.0,0.0,0.0
1,278.299988,98.935997,59.23,6705.120117,90.124001,323.364,4.04,16.43,100.000000,0.0,0.0,0.0,0.0,0.0,0.0,0.0
2,277.000000,98.935997,59.23,6705.120117,90.124001,323.364,4.04,16.43,97.777795,0.0,0.0,0.0,0.0,0.0,0.0,0.0
3,275.799988,98.935997,59.23,6705.120117,90.124001,323.364,4.04,16.43,95.664472,0.0,0.0,0.0,0.0,0.0,0.0,0.0
4,274.200012,98.935997,59.23,6705.120117,90.124001,323.364,4.04,16.43,92.784982,0.0,0.0,0.0,0.0,0.0,0.0,0.0


In [9]:
df_model = df_model.fillna(method="ffill").fillna(method="bfill")

C:\Users\kisho\AppData\Local\Temp\ipykernel_24056\1909595990.py:1: FutureWarning: DataFrame.fillna with 'method' is deprecated and will raise in a future version. Use obj.ffill() or obj.bfill() instead.
  df_model = df_model.fillna(method="ffill").fillna(method="bfill")


In [10]:
train_size = int(len(df_model) * 0.7)
val_size = int(len(df_model) * 0.15)

train = df_model[:train_size]
val = df_model[train_size:train_size + val_size]
test = df_model[train_size + val_size:]

len(train), len(val), len(test)

(4433, 950, 951)

In [11]:
from sklearn.preprocessing import MinMaxScaler

scaler = MinMaxScaler()

scaled_train = scaler.fit_transform(train)
scaled_val = scaler.transform(val)
scaled_test = scaler.transform(test)

In [12]:
train_chronos = train['GOLD_CLOSE'].values
val_chronos = val['GOLD_CLOSE'].values
test_chronos = test['GOLD_CLOSE'].values

In [15]:
import numpy as np

np.save(r"C:\Users\kisho\OneDrive\Desktop\GOLD\data\processed\train_chronos.npy", train_chronos)
np.save(r"C:\Users\kisho\OneDrive\Desktop\GOLD\data\processed\val_chronos.npy", val_chronos)
np.save(r"C:\Users\kisho\OneDrive\Desktop\GOLD\data\processed\test_chronos.npy", test_chronos)

In [16]:
np.save(r"C:\Users\kisho\OneDrive\Desktop\GOLD\data\processed\train_timesfm.npy", scaled_train)
np.save(r"C:\Users\kisho\OneDrive\Desktop\GOLD\data\processed\val_timesfm.npy", scaled_val)
np.save(r"C:\Users\kisho\OneDrive\Desktop\GOLD\data\processed\test_timesfm.npy", scaled_test)

In [17]:
import joblib
joblib.dump(scaler, r"C:\Users\kisho\OneDrive\Desktop\GOLD\data\processed\scaler.pkl")

['C:\\Users\\kisho\\OneDrive\\Desktop\\GOLD\\data\\processed\\scaler.pkl']